# Feature Selection

## Overview
This notebook compares several feature-selection strategies on the prepared lung cancer feature tables and keeps the workflow split into small, readable steps.

In [2]:
import pandas as pd
import re
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import warnings

from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import StratifiedKFold, cross_val_score, train_test_split
from sklearn.impute import SimpleImputer
from sklearn.feature_selection import RFECV, VarianceThreshold, SelectKBest, mutual_info_classif, SequentialFeatureSelector
from sklearn.linear_model import Lasso
from sklearn.base import BaseEstimator
from sklearn.metrics import accuracy_score, recall_score, precision_score, f1_score, roc_auc_score, roc_curve, auc, confusion_matrix
from sklearn.neighbors import KNeighborsClassifier

with warnings.catch_warnings():
    warnings.simplefilter("ignore")

%run ../01_preprocessing/pre_processing2.ipynb

Missing values in Diagnosis at the Patient Level
0=Unknown
1=benign or non-malignant disease
2= malignant, primary lung cancer
3 = malignant metastatic
: 847
Missing values in Diagnosis Method
0 = unknown
1 = review of radiological images to show 2 years of stable nodule
2 = biopsy
3 = surgical resection
4 = progression or response: 847
Missing values in Primary tumor site for metastatic disease: 847
Missing values in Diagnosis Nodule 1: 865
Missing values in Diagnosis Method Nodule 1: 865
Missing values in Diagnosis Nodule 2: 954
Missing values in Diagnosis Method Nodule 2: 954
Missing values in Diagnosis Nodule 3: 966
Missing values in Diagnosis Method Nodule 3: 966
Missing values in Diagnosis Nodule 4: 966
Missing values in Diagnosis Method Nodule 4: 966
Missing values in Diagnosis Nodule 5: 966
Missing values in Diagnosis Method Nodule 5: 966
Missing values in Diagnosis at the Patient Level
0=Unknown
1=benign or non-malignant disease
2= malignant, primary lung cancer
3 = malignant 

## Recursive Feature Elimination (RFE)
RFE ranks features by repeatedly training a Random Forest and keeping the subset that gives the best cross-validated accuracy.

In [ ]:
def rfe(dataframe, target_column, num_features_to_select=5):
    X = dataframe.drop(columns=[target_column])
    y = dataframe[target_column]

    model = RandomForestClassifier()
    selector = RFECV(model, step=1, cv=5, scoring='accuracy', min_features_to_select=num_features_to_select)
    selector.fit(X, y)

    return X.columns[selector.support_]

low_rfe = rfe(low_2d_sd, 'Malignancy')
medium_rfe = rfe(medium_2d_sd, 'Malignancy')
high_rfe = rfe(high_2d_sd, 'Malignancy')

print('Selected Features - sd=0.5:', low_rfe)
print('Selected Features - sd=1.0:', medium_rfe)
print('Selected Features - sd=1.5:', high_rfe)

## Tree-Based Feature Importance
This section keeps features whose Random Forest importance is above a small threshold and compares two cutoffs.

In [ ]:
def feature_select_trees(threshold, dataframe, target_column):
    X = dataframe.drop(columns=[target_column])
    y = dataframe[target_column]

    model = RandomForestClassifier(random_state=42)
    model.fit(X, y)
    selected_features = X.columns[model.feature_importances_ > threshold]

    return selected_features

low_tree_01 = feature_select_trees(0.01, low_2d_sd, 'Malignancy')
medium_tree_01 = feature_select_trees(0.01, medium_2d_sd, 'Malignancy')
high_tree_01 = feature_select_trees(0.01, high_2d_sd, 'Malignancy')

low_tree_02 = feature_select_trees(0.02, low_2d_sd, 'Malignancy')
medium_tree_02 = feature_select_trees(0.02, medium_2d_sd, 'Malignancy')
high_tree_02 = feature_select_trees(0.02, high_2d_sd, 'Malignancy')

print('Threshold 0.01:')
print('Low 2D SD:', low_tree_01)
print('Medium 2D SD:', medium_tree_01)
print('High 2D SD:', high_tree_01)

print('Threshold 0.02:')
print('Low 2D SD:', low_tree_02)
print('Medium 2D SD:', medium_tree_02)
print('High 2D SD:', high_tree_02)

## Variance Threshold
This step keeps only features whose variance is above a fixed cutoff for each prepared dataset.

In [ ]:
X_high_variance1, low_variance_threshold = variance_threshold(0.01, low_2d_sd, 'Malignancy')
X_high_variance2, medium_variance_threshold = variance_threshold(0.01, medium_2d_sd, 'Malignancy')
X_high_variance3, high_variance_threshold = variance_threshold(0.01, high_2d_sd, 'Malignancy')

print('Low Variance Threshold:', low_variance_threshold)
print('Medium Variance Threshold:', medium_variance_threshold)
print('High Variance Threshold:', high_variance_threshold)

## Lasso
Lasso is used here as a sparse linear selector: coefficients that shrink to zero are treated as non-selected features.

In [ ]:
def lasso(dataframe, target_column):
    X = dataframe.drop(columns=[target_column])
    y = dataframe[target_column]

    model = Lasso(alpha=0.01)
    model.fit(X, y)
    return X.columns[model.coef_ != 0]

print('Low Lasso:')
low_lasso = lasso(low_2d_sd, 'Malignancy')
print(low_lasso)

print('Medium Lasso:')
medium_lasso = lasso(medium_2d_sd, 'Malignancy')
print(medium_lasso)

print('High Lasso:')
high_lasso = lasso(high_2d_sd, 'Malignancy')
print(high_lasso)

## Cross-Validation Score Comparison
The final plot compares the mean cross-validation score for each feature-selection strategy so the best-performing groups are visible at a glance.

In [ ]:
feature_sets = [
    'RFE Features sd=0.5',
    'RFE Features sd=1.0',
    'RFE Features sd=1.5',
    'Variance Threshold Features sd=0.5',
    'Variance Threshold Features sd=1.0',
    'Variance Threshold Features sd=1.5',
    'Tree Features sd=0.5 threshold=0.01',
    'Tree Features sd=0.5 threshold=0.02',
    'Tree Features sd=1.0 threshold=0.01',
    'Tree Features sd=1.0 threshold=0.02',
    'Tree Features sd=1.5 threshold=0.01',
    'Tree Features sd=1.5 threshold=0.02',
    'Lasso Features sd=0.5',
    'Lasso Features sd=1.0',
    'Lasso Features sd=1.5',
]

cross_val_scores = [
    cv_scores_low_rfe,
    cv_scores_medium_rfe,
    cv_scores_high_rfe,
    cv_scores_low_vt,
    cv_scores_medium_vt,
    cv_scores_high_vt,
    cv_scores_tree_low01,
    cv_scores_tree_low02,
    cv_scores_tree_medium01,
    cv_scores_tree_medium02,
    cv_scores_tree_high01,
    cv_scores_tree_high02,
    cv_scores_low_lasso,
    cv_scores_medium_lasso,
    cv_scores_high_lasso,
]

plt.figure(figsize=(20, 8))
bars = plt.bar(feature_sets, [score.mean() for score in cross_val_scores], color='#FF7F24', alpha=0.4, label='Cross-Validation Score')
plt.xlabel('Feature Sets')
plt.ylabel('Score')
plt.xticks(rotation=45, ha='right')
plt.legend(loc='best')
plt.title('Cross-Validation Scores')

for bar in bars:
    plt.text(bar.get_x() + bar.get_width() / 2, bar.get_height(), f'{bar.get_height():.2f}', ha='center', va='bottom', fontsize=12, color='black')